In [ ]:
from huggingface_hub import login

# Replace 'your_token_here' with your actual Hugging Face access token
login(token='')#use erad token of HF

In [2]:
# Install Unsloth and its dependencies
!pip install unsloth
!pip install --no-deps trl accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.6 MB/s eta 0:00:0000:010:01
   ━

In [3]:
import torch, math, sqlite3, re, unicodedata
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from unsloth import FastLanguageModel
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from datasets import Dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "kazol196295/llama-3.1-8b-bengali-mental-health-counsellor",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)

# Save references so later cells don't overwrite them
finetuned_model     = model
finetuned_tokenizer = tokenizer

print("✅ Model loaded successfully")


==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Model loaded successfully


In [5]:

from transformers import AutoTokenizer as _AutoTok
count_tokenizer = _AutoTok.from_pretrained("unsloth/Llama-3.1-8B-Instruct")

config.json:   0%|          | 0.00/896 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [7]:
df["q_tokens"] = df["Questions"].apply(lambda x: len(count_tokenizer.encode(str(x))))
df["a_tokens"] = df["Answers"].apply(lambda x: len(count_tokenizer.encode(str(x))))
df["total_tokens"] = df.apply(
    lambda row: len(count_tokenizer.encode(
        f"[INST]\n{row['Questions']}\n[/INST]\nCounselor:\n{row['Answers']}"
    )), axis=1
)
df = df[df["a_tokens"] > 40]
df = df[df["total_tokens"] < 2048]
def format_prompt(row):
    return f"""[INST]
Topic: {row['Topics']}
User: {row['Questions']}
Respond empathetically as a counselor.
[/INST]
Counselor:
{row['Answers']}"""

df["text"] = df.apply(format_prompt, axis=1)
print(f"Filtered dataset size: {len(df)}")

Filtered dataset size: 27388


In [8]:
# Must use same seed=42 as training to get identical test set
full_dataset = Dataset.from_pandas(df)
split1   = full_dataset.train_test_split(test_size=0.05, seed=42)
split2   = split1["test"].train_test_split(test_size=0.10, seed=42)
test_ds  = split2["train"]  # same test set as training notebook
print(f"Test set size: {len(test_ds)}")

Test set size: 1233


In [10]:
test_ds  = split2["train"].select(range(5))  # same test set as training notebook
print(f"Test set size: {len(test_ds)}")

Test set size: 5


In [12]:
def run_nlp_metrics(model, tokenizer, dataset):
    results = []
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
    smooth = SmoothingFunction().method1
    eval_set = dataset.select(range(min(100, len(dataset))))

    print("Generating responses for BLEU / ROUGE...")
    for row in tqdm(eval_set):
        # ✅ Prompt WITHOUT answer — same indentation as training
        prompt = f"""[INST]
                    Topic: {row['Topics']}
                    User: {row['Questions']}
                    Respond empathetically as a counselor.
                    [/INST]
                    Counselor:
                    """
        reference = str(row['Answers']).strip()

        inputs  = tokenizer(prompt, return_tensors="pt",
                            truncation=True, max_length=1024).to("cuda")
        out_ids = model.generate(
            **inputs, max_new_tokens=256, use_cache=True,
            temperature=0.7, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        # Strip prompt tokens, decode only new generated tokens
        gen_ids    = out_ids[0][inputs["input_ids"].shape[1]:]
        prediction = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        r_score = scorer.score(reference, prediction)
        b_score = sentence_bleu([reference.split()], prediction.split(),
                                smoothing_function=smooth)

        results.append({
            "topic":      row['Topics'],
            "input":      row['Questions'],
            "reference":  reference,
            "prediction": prediction,
            "bleu":       b_score,
            "rouge1":     r_score['rouge1'].fmeasure,
            "rougeL":     r_score['rougeL'].fmeasure,
            "timestamp":  datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

    return pd.DataFrame(results)

metrics_df = run_nlp_metrics(finetuned_model, finetuned_tokenizer, test_ds)
print(f"\nAvg BLEU   : {metrics_df['bleu'].mean():.4f}")
print(f"Avg ROUGE-1: {metrics_df['rouge1'].mean():.4f}")
print(f"Avg ROUGE-L: {metrics_df['rougeL'].mean():.4f}")

Generating responses for BLEU / ROUGE...


100%|██████████| 10/10 [02:58<00:00, 17.82s/it]


Avg BLEU   : 0.0088
Avg ROUGE-1: 0.0000
Avg ROUGE-L: 0.0000


In [38]:
print("--- Human Evaluation Sample ---")
# Pick 5 random samples to review
sample_eval = metrics_df.sample(5)

for i, row in sample_eval.iterrows():
    print(f"\n[TOPIC]: {row['topic']}")
    print(f"[USER]: {row['input']}")
    print(f"[MODEL RESPONSE]: {row['prediction']}")
    print(f"[HUMAN REFERENCE]: {row['reference']}")
    print("-" * 30)
    # Manual Task: Rate the MODEL RESPONSE from 1-5 on empathy quality.

--- Human Evaluation Sample ---

[TOPIC]: রাগান্বিত
[USER]: এটা একধরনের ইন্টারনেট ট্রলিং ছিল। তার কোন উপায় ছিল না. আমি মনে করি সে আসলে একটু ভাগ্যবান যে সে বেশি হারায়নি।
[MODEL RESPONSE]: আমি দুঃখিত আপনি এই মধ্য দিয়ে যেতে হয়েছে. এটা ভাল যে আপনি এটা সম্পর্কে খুশি. আমি আশা করি এটি আপনার জন্য একটি ভাল শিক্ষা হয়েছে। আপনি কি কখনও এই ব্যক্তির সাথে সংযোগ করতে চান? আমি আশা করি আপনি এটি করবেন এবং সে এ�
[HUMAN REFERENCE]: উফ! এই সত্যিই আমাকে বিরক্ত. যা ঘটেছে তার জন্য আমি খুবই দুঃখিত। খুব খারাপ তারা সেই লতাগুলোকে ধরতে পারেনি।
------------------------------

[TOPIC]: বিশ্বাস
[USER]: আমি সম্প্রতি প্রথমবারের মতো আমার ছেলেকে বেবিসিটারের সাথে রেখেছি।
[MODEL RESPONSE]: আপনি কি সত্যিই তাকে বিশ্বাস করেন নাকি আপনি যখন কিছু করেন তখন আপনি কখনই আপনার শিশুকে বিশ্বাস করবেন না? আমি মনে করি এটা একটা খারাপ বিষয় না. আপনার বাচ্চারা কি আপনাকে বিশ্বাস করে? আপনি কি কখনও তাদের বিশ্বাস করেছেন যখন আপনি সত্যিই �
[HUMAN REFERENCE]: উঃ আমার মনে আছে প্রথমবার যখন আমি আমার ছেলেকে বেবিসিটারের কাছে রেখেছিলাম। অনেক অনুভূতি। 

In [13]:
def human_eval(metrics_df, num_samples=5):
    print("\n=== Human Evaluation — Rate empathy quality (1=poor, 5=excellent) ===\n")
    sample = metrics_df.sample(min(num_samples, len(metrics_df)), random_state=42)
    scores = []

    for i, (_, row) in enumerate(sample.iterrows(), 1):
        print(f"\nSample {i}/{num_samples}")
        print("─" * 60)
        print(f"[TOPIC]    : {row['topic']}")
        print(f"[USER]     : {row['input']}")
        print(f"[RESPONSE] : {row['prediction']}")
        print("─" * 60)
        while True:
            try:
                score = float(input("Empathy score [1.0 – 5.0]: "))
                if 1.0 <= score <= 5.0:
                    scores.append(score)
                    break
                else:
                    print("Enter a number between 1.0 and 5.0")
            except ValueError:
                print("Invalid input — try again")

    avg = sum(scores) / len(scores) if scores else None
    print(f"\n✅ Average human empathy score: {avg:.2f} / 5.0")
    return avg

avg_human_score = human_eval(metrics_df, num_samples=5)


=== Human Evaluation — Rate empathy quality (1=poor, 5=excellent) ===


Sample 1/5
────────────────────────────────────────────────────────────
[TOPIC]    : উদ্বিগ্ন
[USER]     : আমি প্রতি রাতে দুঃস্বপ্ন দেখতাম।
[RESPONSE] : আমি দুঃখিত আপনি যে মাধ্যমে যাচ্ছে. আপনি কি কখনও একটি বিশেষ প্রতিকার খুঁজে পেয়েছেন? আমি বাজি ধরতে পারি যে আপনি কিছু করতে চান না কিন্তু আপনি এখনও এটি করতে পারেন না বলে মনে হচ্ছে। এটা সব একটি পরিবর্তন আপনি অনুভব করতে
────────────────────────────────────────────────────────────


Empathy score [1.0 – 5.0]:  4



Sample 2/5
────────────────────────────────────────────────────────────
[TOPIC]    : রাগান্বিত
[USER]     : এটা একধরনের ইন্টারনেট ট্রলিং ছিল। তার কোন উপায় ছিল না. আমি মনে করি সে আসলে একটু ভাগ্যবান যে সে বেশি হারায়নি।
[RESPONSE] : আমি আশা করি আপনি এটি নিয়ে একটি সুন্দর মজার সময় কাটাতে পারেন! আমি কিছু না কিছুর জন্য প্রস্তুত থাকব যাতে আমি এটি নিয়ে একটি সুন্দর সময় কাটাতে পারি! আপনি কি সম্প্রতি একটি নতুন গাড়ি কিনেছেন? আপনি এটা কি আপনার প্রথ�
────────────────────────────────────────────────────────────


Empathy score [1.0 – 5.0]:  3



Sample 3/5
────────────────────────────────────────────────────────────
[TOPIC]    : LGBTQ, পারিবারিক দ্বন্দ্ব
[USER]     :  তিনি বলেছিলেন যে তিনি চেষ্টা করবেন এবং তিনি কখনই করেননি। এটি নয় মাস হয়ে গেছে, এবং এটি আমাকে আরও খারাপ করে তুলছে। আজ, তিনি বলেছিলেন যে তিনি যে সিদ্ধান্তই নেবেন আমাকে সম্মান করতে হবে। এটা কি সত্যি? আমি কি চলে যাওয়ার সিদ্ধান্তকে সম্মান করার কথা কারণ সে আমি যা যাচ্ছি তা হ্যান্ডেল করতে পারে না এবং প্রতি রাতে আমাকে কাঁদতে ও চিন্তিত করে রেখে যায়? তিনি ক্রমাগত তার মন পরিবর্তন করছেন যদি তিনি এটি কাজ করতে চান।
[RESPONSE] : আপনার সঙ্গী যে প্রতিশ্রুতি দিয়েছেন তা আপনি কি আশা করেছিলেন?  আপনার সঙ্গী কি আপনাকে বলেছিলেন যে আপনার সম্পর্কের সাথে কীভাবে তিনি কাজ করবেন?  আপনার সঙ্গী যে সিদ্ধান্তগুলি নেয় তার আপনার সাথে কথা বলার সাথে সাথে আপনি কি প্র�
────────────────────────────────────────────────────────────


...


Sample 4/5
────────────────────────────────────────────────────────────
[TOPIC]    : ভয়
[USER]     : সাহসীরা সত্যিই ভাল, আমি জানি না আমাদের রান করার বাজি আছে কিনা
[RESPONSE] : আমি জানি না এই ভাবে কি করতে হবে। আপনি কি পুলিশ ডাকলেন? আমি আশা করি সবকিছু ঠিক আছে। আমি আপনার জন্য আপনার বাড়িতে ফিরে যেতে চাই. আমি আপনাকে শান্ত করার জন্য আমার বিশ্বাস প্রয়োজন. আমি নিশ্চিত যে আপনি কিছু করবেন না যা আপনি পরে �
────────────────────────────────────────────────────────────


Empathy score [1.0 – 5.0]:  4



Sample 5/5
────────────────────────────────────────────────────────────
[TOPIC]    : বিশ্বাস
[USER]     : আমি সম্প্রতি প্রথমবারের মতো আমার ছেলেকে বেবিসিটারের সাথে রেখেছি।
[RESPONSE] : ওহ এটা খুব সুন্দর. তার বয়স কত? তিনি কি তাকে বেবিসিট করতে দিতে সক্ষম হয়েছিলেন? আপনি একটি ভাল সিদ্ধান্ত নিয়েছেন বলে মনে হচ্ছে. আমি বাজি ধরেছি এটি একটি আশ্চর্যজনক অভিজ্ঞতা ছিল! আমি বাজি ধরেছি তারা একসাথে সত্যিই ভাল স�
────────────────────────────────────────────────────────────


Empathy score [1.0 – 5.0]:  3



✅ Average human empathy score: 3.80 / 5.0


In [12]:
import pandas as pd
from tqdm import tqdm
from datetime import datetime

def generate_response(model, tokenizer, topic, question, max_new_tokens=512):
    prompt = f"""[INST]
                    Topic: {topic}
                    User: {question}
                    Respond empathetically as a counselor.
                    [/INST]
                    Counselor:
                    """
    inputs  = tokenizer(prompt, return_tensors="pt",
                        truncation=True, max_length=2048).to("cuda")
    out_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.7,
        do_sample=True,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )
    gen_ids    = out_ids[0][inputs["input_ids"].shape[1]:]
    prediction = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return prediction

# Generate for test set
print("Generating responses...")
records = []

for row in tqdm(test_ds):
    prediction = generate_response(
        finetuned_model, finetuned_tokenizer,
        row['Topics'], row['Questions']
    )
    records.append({
        "topic":           row['Topics'],
        "user_input":      row['Questions'],
        "reference":       str(row['Answers']).strip(),
        "model_response":  prediction,
        "timestamp":       datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

responses_df = pd.DataFrame(records)
print(f"✅ Generated {len(responses_df)} responses")

Generating responses...


100%|██████████| 5/5 [03:02<00:00, 36.59s/it]

✅ Generated 5 responses


In [13]:
import torch, math

def calculate_perplexity_per_sample(model, tokenizer, topic, question, answer):
    """Returns perplexity for a single sample"""
    full_text = f"""[INST]
                    Topic: {topic}
                    User: {question}
                    Respond empathetically as a counselor.
                    [/INST]
                    Counselor:
                    {answer}"""

    inputs    = tokenizer(full_text, return_tensors="pt",
                          truncation=True, max_length=2048).to("cuda")
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs      = model(**inputs)
        shift_logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = input_ids[..., 1:].contiguous()
        loss         = torch.nn.CrossEntropyLoss(reduction="mean")(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )

    ppl = math.exp(loss.item()) if math.isfinite(loss.item()) else float("inf")
    return round(ppl, 2)


print("Calculating perplexity per sample...")
perplexities = []

for row in tqdm(test_ds):
    # Perplexity on REFERENCE answer
    ppl_ref = calculate_perplexity_per_sample(
        finetuned_model, finetuned_tokenizer,
        row['Topics'], row['Questions'], row['Answers']
    )
    # Perplexity on MODEL'S OWN response
    model_resp = responses_df.loc[
        responses_df['user_input'] == row['Questions'], 'model_response'
    ].values
    ppl_model = calculate_perplexity_per_sample(
        finetuned_model, finetuned_tokenizer,
        row['Topics'], row['Questions'],
        model_resp[0] if len(model_resp) > 0 else row['Answers']
    )
    perplexities.append({
        "ppl_reference": ppl_ref,
        "ppl_model":     ppl_model
    })

ppl_df = pd.DataFrame(perplexities)
responses_df['ppl_reference'] = ppl_df['ppl_reference'].values
responses_df['ppl_model']     = ppl_df['ppl_model'].values

print(f"✅ Avg Perplexity (Reference): {responses_df['ppl_reference'].mean():.2f}")
print(f"✅ Avg Perplexity (Model):     {responses_df['ppl_model'].mean():.2f}")

Calculating perplexity per sample...


100%|██████████| 5/5 [00:07<00:00,  1.40s/it]

✅ Avg Perplexity (Reference): 1.76
✅ Avg Perplexity (Model):     2.02


In [14]:
from IPython.display import display, HTML

def show_comparison_table(df, num_rows=None):
    show_df = df.head(num_rows) if num_rows else df

    rows_html = ""
    for i, row in show_df.iterrows():
        rows_html += f"""
        <tr>
            <td style="background:#f0f4ff;font-weight:bold;padding:8px;vertical-align:top;min-width:120px">
                {row['topic']}
            </td>
            <td style="background:#fff8e1;padding:8px;vertical-align:top;min-width:200px">
                {row['user_input']}
            </td>
            <td style="background:#e8f5e9;padding:8px;vertical-align:top;min-width:200px">
                {row['reference']}
            </td>
            <td style="background:#fce4ec;padding:8px;vertical-align:top;min-width:200px">
                {row['model_response']}
            </td>
            <td style="text-align:center;padding:8px;vertical-align:top;background:#f3e5f5">
                {row['ppl_reference']}
            </td>
            <td style="text-align:center;padding:8px;vertical-align:top;background:#e0f7fa">
                {row['ppl_model']}
            </td>
        </tr>
        """

    html = f"""
    <style>
        table {{ border-collapse: collapse; width: 100%; font-size: 13px; font-family: Arial; }}
        th    {{ background: #37474f; color: white; padding: 10px; text-align: left; }}
        tr:hover td {{ filter: brightness(0.97); }}
    </style>
    <table>
        <thead>
            <tr>
                <th>📌 Topic</th>
                <th>💬 User Input</th>
                <th>📖 Reference Answer</th>
                <th>🤖 Model Response</th>
                <th>📊 PPL (Ref)</th>
                <th>📊 PPL (Model)</th>
            </tr>
        </thead>
        <tbody>
            {rows_html}
        </tbody>
    </table>
    <br>
    <b>Summary:</b> 
    Avg PPL Reference = <b>{df['ppl_reference'].mean():.2f}</b> | 
    Avg PPL Model = <b>{df['ppl_model'].mean():.2f}</b> | 
    Total Samples = <b>{len(df)}</b>
    """
    display(HTML(html))

# Show all rows
show_comparison_table(responses_df)

📌 Topic,💬 User Input,📖 Reference Answer,🤖 Model Response,📊 PPL (Ref),📊 PPL (Model)
ভয়,"সাহসীরা সত্যিই ভাল, আমি জানি না আমাদের রান করার বাজি আছে কিনা","আমি নিজেও একজন রেডস ফ্যান........ আমি জানি সাহসীদের তাদের ডিভিশন জেতার ভালো সুযোগ আছে এবং আমি তাদের সাথে একমত, তারা সম্ভবত গভীর প্লে অফের জন্য তৈরি হয়নি।","এটি প্রাণীগুলিকে অনেক চাপের কারণ। আশা করি আপনার খাওয়ার ঘর থেকে বের হওয়ার পর এটি ধরা পড়ে! আপনি কি জানেন? এটা ঠান্ডা. উঃ একটি মঞ্জুর সঙ্গে আচরণ করো!! কিংবদন্তি যখন বলে যে ""ফেরার"" পথের কোনও সৈকতে আপনি বন্য শিশিরে ঢুকলে আপনি একটি কান্নার ভয়ঙ্কর শব্দ শুনতে পাবেন...বাস্তবে!! আমি ভেবেছিলাম এটি একটি মিথ, কিন্তু বিষয়টি একটি বার্তার মতো মনে হয়েছিল..যে আপনি যখন বিভিন্ন ঝড় থেকে সাহস দেখায় তখন তারা আপনাকে আপনার কল্পনার চেয়ে অনে",2.09,2.14
রাগান্বিত,এটা একধরনের ইন্টারনেট ট্রলিং ছিল। তার কোন উপায় ছিল না. আমি মনে করি সে আসলে একটু ভাগ্যবান যে সে বেশি হারায়নি।,উফ! এই সত্যিই আমাকে বিরক্ত. যা ঘটেছে তার জন্য আমি খুবই দুঃখিত। খুব খারাপ তারা সেই লতাগুলোকে ধরতে পারেনি।,"ওহ, অন্তত খুচরা দোকানে থাকতে জিনিস ঠিক করা হয়! ড্রাইভওয়ে ঘটনায় আপনার বাড়িতে ফিরতে খুব সৌভাগ্যবশত। যদি ঈষৎ ঝোঁক-মৃত্যুমুখী হয়? :) আপনি কি সব প্রয়োজনীয় জিনিস খুঁজে পেয়েছেন? এটা ভয়ানক.. সুখী দিন কামনা করি! এই জিনিস আপনাকে সঙ্গে বাণিজ্য মঞ্জুর করতে হবে! আমি নিশ্চিত আপনি পরের বার আপনার ভ্রমণ নিশ্চিত করবেন! কিন্তু যদি না হয়... তারা সবসময় আপনার জন্য এটি সমাধান করতে পারে! আমি যদি সঠিক বড়াই হতাম!! আমার স",1.82,2.0
হতাশ,আগামী সপ্তাহে আশা করছি,আমি তোমার জন্য প্রার্থনা করব! আপনি জ্বলে আউট আগে একটি বিরতি প্রয়োজন.,"ভাল, এটা অন্তত দুর্বল। যেখানে থাকতে চাও? :) আমিই জানি! ধন্যবাদ. ফেরার উঠোন আঘাত করেছে, যদিও আমি একটি ডিম্পে চড়েছি এবং তারপর একটি আঙ্গুল ছেড়ে দিয়েছি (আপনি কি এটি পছন্দ করেন?) - আমি মনে করি সে একটি আঙ্গুলের জন্য আঘাত করেছে! এখন আমি আমার আঙ্গুল মেরামত করছি এবং তার কাছে যাচ্ছি। এটা খুব মৃত্যুদণ্ড মত দেখতে. আমি আশা করি আপনার আঙ্গুলও ঠিক হবে! আমি তোমার জন্য ঈশ্বরের প্রার্থনা করছি!! আমি মনে করি আপনি এই গ্রীষ্মে তার সাথে কাটাতে �",1.69,2.0
সংবেদনশীল,আমি মনে করি আমার এমন একটি শখ খুঁজে বের করা দরকার যা আমাকে ব্যস্ত রাখে!,সাহায্য করতে পারি. আমি জানি আমি আমার শখ ভালোবাসি হাহা.,"হ্যাঁ, নিউ ইয়র্ক থেকে চলে গেছে। পৃষ্ঠ-ঘর অনুভব করা খোলাফেরা? ওড়া! মাঝে মাঝে এমন কিছু ধরনের জিনিস আছে যা আপনি নিজের জন্য শুরু করতে পারেন। ঈশ্বরকে ধন্যবাদ. কাঙ্ক্ষিত সৌভাগ্য!! :)[/INST] 1/৩ শুণতে থাকুন আমি জানি এটি একটি ভাল সমাধান হতে পারে... আমার একটি মনোযোগের সমস্যা আছে যা আমাকে নিজেকে সম্পূর্ণরূপে ফোকাস করতে দেয় না। আমি আপনার সৎ প্রতিক্রিয়ার জন্য আপনাকে ধন্যবাদ জানাচ্ছি। একটি সত্যিই খারাপ পক্ষ - মনোযোগহীনতা। কখনও কখনও এ�",1.65,2.02
হতাশ,আমি বাড়িতে এসে লক্ষ্য করলাম দরজা খোলা ছিল যখন আমার স্ত্রী কাজ করার কথা ছিল। তাই আমি উপরে গিয়েছিলাম এবং সেখানে তারা সেক্স করছে।,"ওহ আমার ঈশ্বর, এটা ভয়ানক. আমি কল্পনা করতে পারি না আপনি কেমন অনুভব করছেন। আমি খুব দুঃখিত হবে","ওহ, অন্তত সে ভালো ফ্টিস! lol চঞ্চল? ঠিক আছে, আপনি সাধারণত তাকে ঘৃণা করেন। আপনি কি তাঁকে ক্ষমা করেছেন? বা আপনি কি কৌতূহল বোধ করেছেন? lol আপনি কি কখনও তাকে বলেছেন যে সে একটি মিস করেছে? :) এটা এখন এক বছর. আমি সঙ্গীর বিয়ে করেছি এবং আমার স্ত্রী আমাকে বলেছিল ""আমি যে সম্পর্কে কথা"" যখন আমি তাকে বলেছিলাম...এটি তার কাছে ছিল..তারা যা করে তা আমার সম্পর্কে নয় কিন্তু যখন আমি কিছু না করে তাদের সম্পর্কে কথা বলি তখন আমার স্ত্রী আমাকে বলে...আমি আপনাকে আ",1.53,1.92


In [14]:
# ── SQLite logging ─────────────────────────────────────────────────────────
conn = sqlite3.connect("experiments.db")
c    = conn.cursor()

c.execute("""CREATE TABLE IF NOT EXISTS LLAMAExperiments (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    model_name TEXT, lora_config TEXT,
    train_loss REAL, val_loss REAL,
    perplexity REAL, avg_bleu REAL,
    avg_rouge1 REAL, avg_rougel REAL,
    avg_human_empathy REAL, timestamp TEXT
)""")

c.execute("""CREATE TABLE IF NOT EXISTS GeneratedResponses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    experiment_id INTEGER, input_text TEXT,
    response_text TEXT, timestamp TEXT
)""")

c.execute("""INSERT INTO LLAMAExperiments
    (model_name, lora_config, train_loss, val_loss, perplexity,
     avg_bleu, avg_rouge1, avg_rougel, avg_human_empathy, timestamp)
    VALUES (?,?,?,?,?,?,?,?,?,?)""", (
    "LLaMA-3.1-8B-Instruct",
    "r16_alpha16_Unsloth",
    None,                              # train_loss lost due to crash
    None,                              # val_loss lost due to crash
    round(ppl_score, 2),
    round(metrics_df['bleu'].mean(),   4),
    round(metrics_df['rouge1'].mean(), 4),
    round(metrics_df['rougeL'].mean(), 4),
    round(avg_human_score, 2) if avg_human_score else None,
    datetime.now().strftime("%Y-%m-%d %H:%M:%S")
))
exp_id = c.lastrowid

for _, row in metrics_df.iterrows():
    c.execute("""INSERT INTO GeneratedResponses
        (experiment_id, input_text, response_text, timestamp)
        VALUES (?,?,?,?)""",
        (exp_id, row['input'], row['prediction'], row['timestamp']))

conn.commit()
conn.close()

# ── Also save to CSV ────────────────────────────────────────────────────────
metrics_df.to_csv("evaluation_results.csv", index=False)

summary = pd.DataFrame([{
    "model":            "LLaMA-3.1-8B-Instruct (LoRA/Unsloth)",
    "perplexity":       round(ppl_score, 2),
    "avg_bleu":         round(metrics_df['bleu'].mean(),   4),
    "avg_rouge1":       round(metrics_df['rouge1'].mean(), 4),
    "avg_rougeL":       round(metrics_df['rougeL'].mean(), 4),
    "avg_human_empathy": round(avg_human_score, 2) if avg_human_score else "N/A"
}])

print("\n" + "="*60)
print("         FINAL EVALUATION SUMMARY")
print("="*60)
print(summary.to_string(index=False))
print("\n✅ Saved: experiments.db  |  evaluation_results.csv")


         FINAL EVALUATION SUMMARY
                               model  perplexity  avg_bleu  avg_rouge1  avg_rougeL  avg_human_empathy
LLaMA-3.1-8B-Instruct (LoRA/Unsloth)        1.72    0.0088         0.0         0.0                3.8

✅ Saved: experiments.db  |  evaluation_results.csv


In [20]:
pd.read_csv("evaluation_results.csv")

,topic,input,reference,prediction,bleu,rouge1,rougeL,timestamp
0,ভয়,"সাহসীরা সত্যিই ভাল, আমি জানি না আমাদের রান করা...",আমি নিজেও একজন রেডস ফ্যান........ আমি জানি সাহ...,আমি জানি না এই ভাবে কি করতে হবে। আপনি কি পুলিশ...,0.010874,0.0,0,2026-03-11 11:25:50
1,রাগান্বিত,এটা একধরনের ইন্টারনেট ট্রলিং ছিল। তার কোন উপায...,উফ! এই সত্যিই আমাকে বিরক্ত. যা ঘটেছে তার জন্য ...,আমি আশা করি আপনি এটি নিয়ে একটি সুন্দর মজার সম...,0.005495,0.0,0,2026-03-11 11:26:08
2,হতাশ,আগামী সপ্তাহে আশা করছি,আমি তোমার জন্য প্রার্থনা করব! আপনি জ্বলে আউট আ...,আমি আশা করি আপনি করবেন. আমি আশা করি আপনি এটি প...,0.003782,0.0,0,2026-03-11 11:26:26
3,সংবেদনশীল,আমি মনে করি আমার এমন একটি শখ খুঁজে বের করা দরক...,সাহায্য করতে পারি. আমি জানি আমি আমার শখ ভালোবা...,একটি শখ খুঁজে পেতে সবসময় সময় লাগে। আমি আশা ক...,0.012615,0.0,0,2026-03-11 11:26:43
4,হতাশ,আমি বাড়িতে এসে লক্ষ্য করলাম দরজা খোলা ছিল যখন...,"ওহ আমার ঈশ্বর, এটা ভয়ানক. আমি কল্পনা করতে পার...",আমি নিশ্চিত নই যে আমি কি করব যদি এটি আমার সাথে...,0.006212,0.0,0,2026-03-11 11:27:01
5,"LGBTQ, পারিবারিক দ্বন্দ্ব",তিনি বলেছিলেন যে তিনি চেষ্টা করবেন এবং তিনি ক...,যদি আপনার স্বামী সম্পর্কের মধ্যে থাকতে চান কি ...,আপনার সঙ্গী যে প্রতিশ্রুতি দিয়েছেন তা আপনি কি...,0.000157,0.0,0,2026-03-11 11:27:19
6,প্রত্যাশিত,এটা কিছু সত্যিই মজা ছিল,আপনি কি এমন কিছু চড়েছেন যা আপনাকে অসুস্থ করেছ...,আপনি কি সেই প্রকল্পের পরে কাজ করেছেন? আপনি কি ...,0.010814,0.0,0,2026-03-11 11:27:37
7,বিশ্বাস,আমি সম্প্রতি প্রথমবারের মতো আমার ছেলেকে বেবিসি...,উঃ আমার মনে আছে প্রথমবার যখন আমি আমার ছেলেকে ব...,ওহ এটা খুব সুন্দর. তার বয়স কত? তিনি কি তাকে ব...,0.026679,0.0,0,2026-03-11 11:27:55
8,উদ্বিগ্ন,আমি প্রতি রাতে দুঃস্বপ্ন দেখতাম।,শুনে দুঃখিত হলাম. আপনার দুঃস্বপ্নের সময় কি ঘটে?,আমি দুঃখিত আপনি যে মাধ্যমে যাচ্ছে. আপনি কি কখন...,0.005356,0.0,0,2026-03-11 11:28:13
9,প্রস্তুত,"বিশ্ব এখন যেভাবে চলছে, আমি গ্রিড বন্ধ করার চেষ...",এটি একটি আকর্ষণীয় জীবন পছন্দ মত শোনাচ্ছে. এটা...,এটা আশ্চর্যজনক! আপনি একটি পরিবার আছে? আপনি কি ...,0.006081,0.0,0,2026-03-11 11:28:30
